# MAMMAL Fine-Tuning

Fine-tune the pretrained MAMMAL model (`ibm/biomed.omics.bl.sm.ma-ted-458m`) for **binary classification** of
molecules (e.g. hit / non-hit) from SMILES.

MAMMAL has no classification head: the molecule is written into a text prompt and the model answers with the
token `<0>` or `<1>`. The score is the probability it gives to `<1>`.

**Environment:** run with the `biomed-mammal` kernel (see [INSTALL.md](../INSTALL.md)).

**Inputs:** a train file and a test file (`.csv` or `.parquet`), each with a SMILES column and a 0/1 label column.

**Metrics** (the same as the MMELON notebooks, so the two models can be compared)
- **hit@K** (hit@200, hit@100, hit@50): rank the test molecules by predicted score; hit@K is how many
  of the top K have label 1. If the test set has fewer than K molecules, all of them are counted.
- Standard classification metrics: AUROC, PR AUC, precision, recall, specificity and loss.

**How the model is selected:** train on the train file for up to `EPOCHS` epochs, scoring the test file after
every epoch. Epochs are ranked by `SELECT_METRICS` (default: hit@200, then hit@100, then hit@50, then AUROC to
break ties), and the best epoch is kept. Training stops early when `PATIENCE` epochs in a row bring no
improvement, and only the best epoch's model is kept on disk.

Because the test file is used to pick the epoch, its score is slightly optimistic.
For an unbiased estimate, evaluate the final model on a separate held-out file.

For k-fold cross-validation, use [MAMMAL_finetuning_cv.ipynb](MAMMAL_finetuning_cv.ipynb) instead.

**Full fine-tuning or LoRA:** `USE_LORA = False` trains every weight; `True` freezes the model and trains small
added matrices instead (about 0.2% of the weights, far less memory). Either way the saved model is a plain
MAMMAL model, so inference is the same.

**Output** (in `aircheck/output/`, which is gitignored):
```
output/
└── <DATASET>/finetuned/<RUN_NAME>_<timestamp>/
    ├── config.json                      # settings used for this run
    ├── data/                            # the rows actually used (train.csv, val.csv)
    ├── logs/metrics.csv                 # metrics per epoch, as Lightning wrote them
    ├── metrics_per_epoch.csv            # the same, one tidy row per epoch
    ├── selection.json                   # which epoch was chosen and its metrics
    └── best_model/                      # the fine-tuned model and its tokenizer (~1.8 GB)
```

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [ ]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/
DATASET = "sample-data"

# Name of this run; outputs go to output/<DATASET>/finetuned/<RUN_NAME>_<timestamp>/
RUN_NAME = "mammal-final"

# ── Data ──────────────────────────────────────────────────────────────────────
# The MMELON sample data, so both models train and test on identical molecules.
TRAIN_PATH = Path("../../../biomed-multi-view/aircheck/sample_data/train.parquet")
TEST_PATH = Path("../../../biomed-multi-view/aircheck/sample_data/test.parquet")

SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm/biomed.omics.bl.sm.ma-ted-458m"  # Hugging Face model ID

# ── LoRA ──────────────────────────────────────────────────────────────────────
# False: train every weight (needs ~15 GB of GPU memory).
# True:  freeze the model and train small added matrices instead (~4 GB, about 0.2% of the weights).
# Either way the saved model is a normal MAMMAL model: with LoRA the matrices are merged back in at
# the end, so inference is identical.
USE_LORA = False
LORA_R = 8                      # size of the added matrices; larger = more capacity, more memory
LORA_ALPHA = 16                 # how strongly they count; usually 2 x LORA_R
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q", "v"]  # attention query and value layers, the usual choice for T5

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 50                # maximum number of epochs
PATIENCE = 5               # stop after this many epochs without improvement
BATCH_SIZE = 16
LEARNING_RATE = 1e-5       # with USE_LORA, 1e-4 to 3e-4 usually works better
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1      # share of all training steps spent warming the learning rate up
SEED = 42

# Balance the loss between the classes: label 0 gets weight 1, label 1 gets (number of 0s) / (number of 1s),
# counted on the training data only (in CV: on each fold's own training rows). False = unweighted loss.
BALANCE_CLASSES = True

# Prompt lengths, in tokens. A SMILES longer than MAX_SMILES_TOKENS is cut.
MAX_SMILES_TOKENS = 300
MAX_PROMPT_TOKENS = 320

# hit@K = number of label-1 molecules among the K highest-scored test molecules
HIT_KS = [200, 100, 50]

# Model selection, in priority order: the first metric decides, later ones break ties.
# Choose from hit@K (for K in HIT_KS), "rocauc", "pr_auc", "precision", "recall", "specificity", "loss".
SELECT_METRICS = ["hit@200", "hit@100", "hit@50", "rocauc"]

MAX_PER_CLASS = None      # e.g. 100 for a quick test run; None uses all data
NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes

In [ ]:
import json
import shutil
from datetime import datetime

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "mammal").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-alignment repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
RUN_DIR = OUTPUT_DIR / DATASET / "finetuned" / f"{RUN_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DATA_DIR = RUN_DIR / "data"
BEST_MODEL_DIR = RUN_DIR / "best_model"
TOKENIZER_SOURCE = BASE_MODEL

for path in (TRAIN_PATH, TEST_PATH):
    assert Path(path).exists(), f"File not found: {path}"

# Only the best epoch's model (~1.8 GB) is kept, overwritten when a later epoch is better
needed_gb = 1.8 * 2
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > needed_gb + 2, f"Need ~{needed_gb:.0f} GB free for the model, only {free_gb:.0f} GB available"

RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = dict(
    train_path=str(TRAIN_PATH), test_path=str(TEST_PATH),
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL,
    use_lora=USE_LORA, lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODULES,
    epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, warmup_fraction=WARMUP_FRACTION, seed=SEED,
    hit_ks=HIT_KS, select_metrics=SELECT_METRICS, balance_classes=BALANCE_CLASSES,
    max_smiles_tokens=MAX_SMILES_TOKENS, max_prompt_tokens=MAX_PROMPT_TOKENS,
    max_per_class=MAX_PER_CLASS,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Repo root  : {REPO_ROOT}")
print(f"Run folder : {RUN_DIR}")
print(f"Training   : {'LoRA' if USE_LORA else 'all weights'}")
print(f"Disk       : ~{needed_gb:.1f} GB needed, {free_gb:.0f} GB free")
print(f"Device     : {DEVICE}")

## 2. Prepare data

Read both files, keep only the SMILES and label columns, drop empty rows, and save the rows actually used
to the run's `data/` folder so the run can be reproduced.

In [ ]:
import numpy as np
import pandas as pd
import torch
from functools import partial
from torch.utils.data import DataLoader, Dataset

import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger
from sklearn.metrics import average_precision_score, roc_auc_score
from peft import get_peft_model_state_dict, set_peft_model_state_dict

from fuse.data.utils.collates import CollateDefault
from fuse.dl.lightning.pl_module import LightningModuleDefault
from mammal.keys import *
from mammal.lora import get_lora_model
from mammal.losses import LossHead
from mammal.lr_schedulers import cosine_annealing_with_warmup_lr_scheduler
import pyarrow.parquet as pq
def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)
def load_split(path: Path) -> pd.DataFrame:
    df = read_table(path, [SMILES_COL, LABEL_COL]).rename(columns={SMILES_COL: "smiles", LABEL_COL: "label"})
    n_before = len(df)
    df = df.dropna()
    df = df[df["smiles"].astype(str).str.strip() != ""]
    if len(df) < n_before:
        print(f"{path.name}: dropped {n_before - len(df)} rows with empty SMILES or label")
    df["label"] = df["label"].astype(int)
    assert set(df["label"].unique()) <= {0, 1}, f"{path} labels must be 0/1"
    return df.reset_index(drop=True)


def limit_per_class(df: pd.DataFrame, n: int) -> pd.DataFrame:
    parts = [g.sample(n=min(n, len(g)), random_state=SEED) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)


def positive_weight(labels) -> float:
    # Weight for label 1 in the loss: (number of 0s) / (number of 1s), or 1.0 when BALANCE_CLASSES is off
    if not BALANCE_CLASSES:
        return 1.0
    n_neg, n_pos = int((labels == 0).sum()), int((labels == 1).sum())
    assert n_neg > 0 and n_pos > 0, "training data needs both labels to balance the classes"
    return round(n_neg / n_pos, 4)


splits, summary = {}, []
for name, path, out_name in [("train", TRAIN_PATH, "train.csv"), ("val", TEST_PATH, "val.csv")]:
    df = load_split(path)
    if MAX_PER_CLASS is not None:
        df = limit_per_class(df, MAX_PER_CLASS)
    df.to_csv(RUN_DATA_DIR / out_name, index=False)
    splits[name] = df
    summary.append(dict(split=name, samples=len(df), positives=int(df["label"].sum()),
                        positive_pct=round(100 * df["label"].mean(), 2)))

POS_WEIGHT = positive_weight(splits["train"]["label"])
config["positive_weight"] = POS_WEIGHT
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
print(f"Loss weight for label 1: {POS_WEIGHT}  (label 0 is always 1.0)")
pd.DataFrame(summary)

## 3. Load the tokenizer

The pretrained model is downloaded from Hugging Face the first time (about 1.8 GB), then cached.

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.model import Mammal

tokenizer_op = ModularTokenizerOp.from_pretrained(TOKENIZER_SOURCE)

NEG_TOKEN_ID = tokenizer_op.get_token_id("<0>")
POS_TOKEN_ID = tokenizer_op.get_token_id("<1>")
PAD_TOKEN_ID = tokenizer_op.get_token_id("<PAD>")
print(f"Tokenizer loaded. Answer tokens: <0> = {NEG_TOKEN_ID}, <1> = {POS_TOKEN_ID}")

## 4. Prompts and data loaders

Each molecule becomes a prompt the model reads and an answer it should write:

```
prompt:  <@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>
         <SEQUENCE_NATURAL_START>CCO<SEQUENCE_NATURAL_END><EOS>
answer:  <SENTINEL_ID_0><1><EOS>
```

`token_weights` carries the class balancing: the answer of a label-1 molecule is weighted by `POS_WEIGHT`,
and ignored positions get 0.

In [ ]:
IGNORE_INDEX = -100
TOKEN_WEIGHTS_KEY = "data.token_weights"
MAX_ANSWER_TOKENS = 4
ANSWER_POSITION = 1  # the decoder writes <SENTINEL_ID_0> first, then the answer


def encoder_prompt(smiles: str) -> str:
    """The text MAMMAL reads for one molecule."""
    return (
        f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0>"
        f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
        f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={MAX_SMILES_TOKENS}>"
        f"<SEQUENCE_NATURAL_START>{smiles}<SEQUENCE_NATURAL_END><EOS>"
    )


def prepare_sample(sample_dict: dict, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True) -> dict:
    """Tokenize one sample: the prompt, and (for training) the answer and its loss weight."""
    sample_dict[ENCODER_INPUTS_STR] = encoder_prompt(sample_dict["data.smiles"])
    tokenizer_op(
        sample_dict=sample_dict,
        key_in=ENCODER_INPUTS_STR,
        key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
        key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_PROMPT_TOKENS,
    )
    for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    if not with_label:
        return sample_dict

    label = int(sample_dict["data.label"])

    # What the model should write
    sample_dict[LABELS_STR] = f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=LABELS_STR,
        key_out_tokens_ids=LABELS_TOKENS, key_out_attention_mask=LABELS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (LABELS_TOKENS, LABELS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])
    # Padding must not count towards the loss
    sample_dict[LABELS_TOKENS][sample_dict[LABELS_TOKENS] == PAD_TOKEN_ID] = IGNORE_INDEX

    # What the decoder is fed during training (teacher forcing)
    sample_dict[DECODER_INPUTS_STR] = f"<@TOKENIZER-TYPE=SMILES><DECODER_START><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=DECODER_INPUTS_STR,
        key_out_tokens_ids=DECODER_INPUTS_TOKENS, key_out_attention_mask=DECODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (DECODER_INPUTS_TOKENS, DECODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    # Class balancing: weight every counted position of this sample, 0 where the loss is ignored
    weight = pos_weight if label == 1 else 1.0
    sample_dict[TOKEN_WEIGHTS_KEY] = weight * (sample_dict[LABELS_TOKENS] != IGNORE_INDEX).float()
    return sample_dict


class MoleculeDataset(Dataset):
    """One row per molecule, from a DataFrame with 'smiles' and (for training) 'label' columns."""

    def __init__(self, df, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True):
        self.smiles = df["smiles"].tolist()
        self.labels = df["label"].tolist() if with_label else [0] * len(df)
        self.tokenizer_op, self.pos_weight, self.with_label = tokenizer_op, pos_weight, with_label

    def __len__(self) -> int:
        return len(self.smiles)

    def __getitem__(self, idx: int) -> dict:
        sample = {"data.sample_id": idx, "data.smiles": self.smiles[idx], "data.label": self.labels[idx]}
        return prepare_sample(sample, self.tokenizer_op, self.pos_weight, self.with_label)


def make_collate():
    """Trim the padding each batch does not need (10-20% faster)."""
    return CollateDefault(special_handlers_keys={
        ENCODER_INPUTS_TOKENS: partial(CollateDefault.crop_padding, pad_token_id=PAD_TOKEN_ID),
        ENCODER_INPUTS_ATTENTION_MASK: partial(CollateDefault.crop_padding, pad_token_id=False),
    })
loaders = {}
for name, shuffle in [("train", True), ("val", False)]:
    loaders[name] = DataLoader(
        MoleculeDataset(splits[name], tokenizer_op, pos_weight=POS_WEIGHT),
        batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=make_collate(),
        num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
    )

batch = next(iter(loaders["val"]))
print(f"{len(loaders['train'])} train batches, {len(loaders['val'])} validation batches "
      f"(batch size {BATCH_SIZE})")
print(f"Prompt length in this batch: {batch[ENCODER_INPUTS_TOKENS].shape[1]} tokens")

## 5. Build the model

`LightningModuleDefault` is MAMMAL's own training module. The subclass below adds the metrics the MMELON
notebooks report, computed from the probability the model gives to `<1>` at the answer position.

In [ ]:
def compute_metrics(labels, scores, hit_ks=None, threshold=0.5) -> dict:
    """hit@K, AUROC, PR AUC and the threshold metrics, as the MMELON notebooks report them."""
    hit_ks = HIT_KS if hit_ks is None else hit_ks
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(labels[top[:k]].sum()) for k in hit_ks}
    pred = scores > threshold
    tp, fp = int((pred & (labels == 1)).sum()), int((pred & (labels == 0)).sum())
    tn, fn = int((~pred & (labels == 0)).sum()), int((~pred & (labels == 1)).sum())
    both_classes = len(set(labels.tolist())) == 2
    metrics.update(
        rocauc=float(roc_auc_score(labels, scores)) if both_classes else float("nan"),
        pr_auc=float(average_precision_score(labels, scores)) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(labels) if len(labels) else float("nan"),
    )
    return metrics
def build_model():
    """The pretrained model, wrapped in LoRA when USE_LORA is on."""
    model = Mammal.from_pretrained(BASE_MODEL)
    if USE_LORA:
        model = get_lora_model(model, r=LORA_R, lora_alpha=LORA_ALPHA,
                               lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES)
    else:
        print(f"Full fine-tuning: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M trainable parameters")
    return model


def save_model(model, path: Path, lora_state: dict | None = None):
    """Save as a plain MAMMAL model, so inference is the same with and without LoRA.

    With LoRA the added matrices are merged into the weights first; `lora_state` restores the best
    epoch's matrices before merging (the model in memory holds the last epoch's)."""
    path.mkdir(parents=True, exist_ok=True)
    if USE_LORA:
        if lora_state is not None:
            set_peft_model_state_dict(model, lora_state)
        merged = model.merge_and_unload()    # gives back a plain Mammal
        merged._save_pretrained(str(path))
    else:
        model._save_pretrained(str(path))
    tokenizer_op.save_pretrained(str(path / "tokenizer"))
class HitAtKModule(LightningModuleDefault):
    """MAMMAL's training module, plus the metrics the MMELON notebooks report.

    The score for a molecule is the probability the model gives to `<1>` at the answer position,
    against `<0>`: the same number the inference notebook computes."""

    @property
    def model(self):
        """The MAMMAL model inside the module (fuse stores it as `_model`)."""
        return self._model

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        self._val_scores, self._val_labels = [], []

    def validation_step(self, batch_dict, batch_idx, dataloader_idx=0):
        super().validation_step(batch_dict, batch_idx, dataloader_idx)
        probs = batch_dict[SCORES][:, ANSWER_POSITION, :]          # (batch, vocabulary)
        pos, neg = probs[:, POS_TOKEN_ID], probs[:, NEG_TOKEN_ID]
        self._val_scores.append((pos / (pos + neg + 1e-8)).detach().float().cpu())
        answers = batch_dict[LABELS_TOKENS][:, ANSWER_POSITION]
        self._val_labels.append((answers == POS_TOKEN_ID).detach().int().cpu())

    def on_validation_epoch_end(self):
        super().on_validation_epoch_end()
        scores = torch.cat(self._val_scores).numpy()
        labels = torch.cat(self._val_labels).numpy()
        for name, value in compute_metrics(labels, scores).items():
            self.log(f"val_{name}", float(value))
        # The loss Lightning logged under its own name, copied to the val_ name the selection uses
        logged = self.trainer.callback_metrics.get("validation.losses.total_loss")
        self.log("val_loss", float(logged) if logged is not None else float("nan"))


def make_optimizers(total_steps: int):
    """AdamW with a warmup then a cosine decay."""
    warmup_steps = max(1, int(WARMUP_FRACTION * total_steps))

    def configure(module):
        opt = torch.optim.AdamW([p for p in module.trainer.model.parameters() if p.requires_grad],
                                lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        sched = cosine_annealing_with_warmup_lr_scheduler(
            opt, T_max=max(1, total_steps - warmup_steps), num_warmup_steps=warmup_steps, eta_min_factor=0.1)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "step"}}

    return configure, warmup_steps


LOSSES = lambda: {"ce": LossHead(loss_type="ce", sample_token_weights_key=TOKEN_WEIGHTS_KEY)}
pl.seed_everything(SEED, workers=True)

model = build_model()
steps_per_epoch = len(loaders["train"])
configure_optimizers, warmup_steps = make_optimizers(steps_per_epoch * EPOCHS)

lightning_module = HitAtKModule(
    model=model, losses=LOSSES(), optimizers_and_lr_schs=configure_optimizers, model_dir=str(RUN_DIR),
)
print(f"{steps_per_epoch * EPOCHS} training steps ({steps_per_epoch} per epoch), "
      f"{warmup_steps} of them warming up")

## 6. Train and select the best epoch

In [ ]:
allowed = [f"hit@{k}" for k in HIT_KS] + ["rocauc", "pr_auc", "precision", "recall", "specificity",
                                          "accuracy", "loss"]
assert set(SELECT_METRICS) <= set(allowed), f"SELECT_METRICS must come from {allowed}"


class KeepBestEpoch(Callback):
    """After each epoch: remember it if it is the best so far by SELECT_METRICS, and stop after
    PATIENCE epochs without improvement.

    Without LoRA the model (~1.8 GB) is written to disk each time it improves. With LoRA only the
    added matrices are kept in memory (a few MB) and merged into the model after training."""

    def __init__(self, path: Path):
        self.path, self.best_key, self.best_epoch, self.wait = path, None, None, 0
        self.lora_state = None

    def on_validation_end(self, trainer, pl_module):
        logged = trainer.callback_metrics
        # Larger is better; loss is negated so a lower loss ranks higher. Ties keep the earlier epoch.
        key = tuple(float(logged[f"val_{m}"]) * (-1 if m == "loss" else 1) for m in SELECT_METRICS)
        if self.best_key is None or key > self.best_key:
            self.best_key, self.best_epoch, self.wait = key, trainer.current_epoch, 0
            if USE_LORA:
                self.lora_state = {k: v.detach().cpu().clone()
                                   for k, v in get_peft_model_state_dict(pl_module.model).items()}
            else:
                save_model(pl_module.model, self.path)
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                print(f"Stopping early after epoch {trainer.current_epoch}: "
                      f"no improvement for {PATIENCE} epochs")
                trainer.should_stop = True

    def finish(self, pl_module):
        """Called after training: with LoRA, merge the best matrices in and save."""
        if USE_LORA:
            save_model(pl_module.model, self.path, lora_state=self.lora_state)
keeper = KeepBestEpoch(BEST_MODEL_DIR)
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=[keeper],
    logger=CSVLogger(save_dir=RUN_DIR, name="logs", version=""),
    enable_checkpointing=False,
    default_root_dir=RUN_DIR,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    num_sanity_val_steps=0,
    enable_model_summary=True,
)

In [ ]:
%%time
trainer.fit(lightning_module, loaders["train"], loaders["val"])
keeper.finish(lightning_module)   # with LoRA: merge the best matrices in and save

In [ ]:
# Test metrics per epoch
logged = pd.read_csv(RUN_DIR / "logs" / "metrics.csv")
metrics = logged.dropna(subset=["val_loss"])
metric_cols = [c for c in metrics.columns if c.startswith("val_")]
metrics = metrics[["epoch"] + metric_cols].rename(columns=lambda c: c.removeprefix("val_"))
metrics = metrics.groupby("epoch", as_index=False).first()  # one row per epoch
metrics["epoch"] = metrics["epoch"].astype(int)

# hit@K first (in HIT_KS order), then the standard metrics
hits = [f"hit@{k}" for k in HIT_KS]
metric_names = hits + [c for c in metrics.columns if c not in hits and c != "epoch"]
metrics = metrics[["epoch"] + metric_names]
metrics.to_csv(RUN_DIR / "metrics_per_epoch.csv", index=False)

# Check: ranking all epochs afterwards picks the same epoch as the choice made during training
ranked = metrics.sort_values(SELECT_METRICS, ascending=[m == "loss" for m in SELECT_METRICS], kind="mergesort")
best_epoch = keeper.best_epoch
assert int(ranked.iloc[0]["epoch"]) == best_epoch, "epoch chosen during training differs from the ranking"
best_metrics = metrics.set_index("epoch").loc[best_epoch]

epochs_run = len(metrics)
selection = dict(
    select_metrics=SELECT_METRICS,
    best_epoch=best_epoch,
    epochs_trained=epochs_run,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    stopped_early=epochs_run < EPOCHS,
    use_lora=USE_LORA,
    test_metrics={m: float(best_metrics[m]) for m in metric_names},
    model=str(BEST_MODEL_DIR),
)
(RUN_DIR / "selection.json").write_text(json.dumps(selection, indent=2))

print(f"Trained {epochs_run} of up to {EPOCHS} epochs{' (stopped early)' if epochs_run < EPOCHS else ''}")
print(f"Selected epoch {best_epoch}:")
for m in SELECT_METRICS:
    print(f"  {m:12s} {best_metrics[m]:.4f}")
print(f"Fine-tuned model: {BEST_MODEL_DIR}\n")
display(metrics.round(4))

In [ ]:
# Files produced by this run
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):60s} {path.stat().st_size / 1e6:8.2f} MB")